# 🏥 MedFusionNet : Auto-Setup & Heavy Training (Standalone Colab)

Ce notebook est **entièrement autonome**. Il va reconstruire tout le code source du projet, installer les dépendances, et lancer l'entraînement sur GPU NVIDIA.

### 📋 Instructions Rapides :
1. **GPU** : Allez dans **Modifier > Paramètres du notebook** et assurez-vous que l'accélérateur est **GPU** (T4, V100 ou A100).
2. **Exécution** : Cliquez sur **Environnement d'exécution > Tout exécuter**.
3. **Kaggle** : À l'étape 3, uploadez votre fichier `kaggle.json` quand on vous le demandera.

## 1️⃣ Génération du Code Source
Cette cellule recrée tous les fichiers Python nécessaires au projet.

In [ ]:
import os

# Création des répertoires de base
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("Reconstruction du projet...")

In [ ]:
%%writefile requirements.txt
torch>=2.0.0
torchvision>=0.15.0
timm>=0.9.0
Pillow>=9.0.0
opencv-python-headless>=4.8.0, <4.9.0
numpy>=1.24.0
scipy>=1.10.0
scikit-learn>=1.2.0
matplotlib>=3.7.0
tqdm>=4.65.0
pandas>=2.0.0

In [ ]:
%%writefile preprocessing.py
import cv2
import numpy as np
from PIL import Image
import torch
from torchvision import transforms
from typing import Tuple, Optional

IMG_SIZE      = 384
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
CLAHE_CLIP_LIMIT    = 2.0
CLAHE_TILE_GRID     = (8, 8)

class CLAHETransform:
    def __init__(self, clip_limit=CLAHE_CLIP_LIMIT, tile_grid_size=CLAHE_TILE_GRID):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
    def __call__(self, img):
        img_gray = np.array(img.convert("L"))
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_clahe = clahe.apply(img_gray)
        img_rgb = cv2.cvtColor(img_clahe, cv2.COLOR_GRAY2RGB)
        return Image.fromarray(img_rgb)

class ZScoreNormalize:
    def __init__(self, mean=IMAGENET_MEAN, std=IMAGENET_STD, per_image=False):
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std  = torch.tensor(std).view(3, 1, 1)
        self.per_image = per_image
    def __call__(self, tensor):
        if self.per_image:
            mean = tensor.mean(dim=[1, 2], keepdim=True)
            std  = tensor.std(dim=[1, 2], keepdim=True).clamp(min=1e-6)
            return (tensor - mean) / std
        return (tensor - self.mean) / self.std

def build_transforms(mode="train", img_size=IMG_SIZE, use_clahe=True, per_image_norm=False):
    initial = [CLAHETransform()] if use_clahe else []
    if mode == "train":
        augmentations = [
            transforms.Resize((img_size + 20, img_size + 20)),
            transforms.RandomCrop((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
        ]
    else:
        augmentations = [
            transforms.Resize((img_size, img_size)),
            transforms.CenterCrop((img_size, img_size)),
        ]
    post = [transforms.ToTensor(), ZScoreNormalize(per_image=per_image_norm)]
    return transforms.Compose(initial + augmentations + post)

class CXRPreprocessor:
    def __init__(self, img_size=IMG_SIZE, use_clahe=True):
        self.transform = build_transforms(mode="test", img_size=img_size, use_clahe=use_clahe)
    def __call__(self, image_path):
        img = Image.open(image_path).convert("RGB")
        return self.transform(img).unsqueeze(0)

In [ ]:
%%writefile dataset.py
import os
from pathlib import Path
from typing import Optional, Dict, Tuple, List
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from preprocessing import build_transforms

CLASS_TO_IDX = {"NORMAL": 0, "PNEUMONIA": 1}
class PneumoniaDataset(Dataset):
    def __init__(self, root_dir, mode="train", img_size=384, use_clahe=True, bbox_file=None):
        self.root_dir = Path(root_dir)
        self.mode = mode
        self.transform = build_transforms(mode=mode, img_size=img_size, use_clahe=use_clahe)
        self.samples = []
        for class_name, label in CLASS_TO_IDX.items():
            class_dir = self.root_dir / class_name
            if class_dir.exists():
                for fp in class_dir.glob("*.jp*g"):
                    self.samples.append((fp, label))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        return {"image": self.transform(img), "label": torch.tensor(label, dtype=torch.long), "path": str(img_path), "bbox": torch.zeros(4)}
    def get_class_weights(self):
        n_total = len(self.samples)
        n_normal = sum(1 for _, l in self.samples if l == 0)
        n_pneumo = n_total - n_normal
        w_n = n_total/(2*n_normal) if n_normal > 0 else 0
        w_p = n_total/(2*n_pneumo) if n_pneumo > 0 else 0
        return torch.tensor([w_n if l==0 else w_p for _, l in self.samples])

def build_dataloaders(data_dir, batch_size=16, img_size=384, num_workers=4, use_clahe=True, use_weighted_sampler=True, bbox_file=None, pin_memory=True):
    loaders = {}
    for mode in ("train", "val", "test"):
        if (Path(data_dir)/mode).exists():
            ds = PneumoniaDataset(Path(data_dir)/mode, mode, img_size, use_clahe)
            sampler = WeightedRandomSampler(ds.get_class_weights(), len(ds), replacement=True) if mode=="train" and use_weighted_sampler else None
            loaders[mode] = DataLoader(ds, batch_size=batch_size, shuffle=(sampler is None and mode=="train"), sampler=sampler, num_workers=num_workers, pin_memory=pin_memory)
    return loaders

In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import densenet121, DenseNet121_Weights
import timm

FUSION_DIM = 512
class LocalBranch(nn.Module):
    def __init__(self, pretrained=True, freeze_until=0):
        super().__init__()
        backbone = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = backbone.features
        self.projection = nn.Sequential(nn.Conv2d(1024, FUSION_DIM, 1), nn.BatchNorm2d(FUSION_DIM), nn.ReLU(True))
    def forward(self, x): return self.projection(F.relu(self.features(x), True))

class GlobalBranch(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.swin = timm.create_model("swinv2_cr_tiny_ns_224.sw_in1k", pretrained=pretrained, img_size=384, num_classes=0, global_pool="")
        self.projection = nn.Sequential(nn.Conv2d(768, FUSION_DIM, 1), nn.BatchNorm2d(FUSION_DIM), nn.ReLU(True))
    def forward(self, x):
        feat = self.swin.forward_features(x)
        if feat.dim() == 3: # (B, N, C)
             B, N, C = feat.shape; H = W = int(N**0.5); feat = feat.permute(0,2,1).reshape(B,C,H,W)
        elif feat.dim() == 4 and feat.shape[3] == 768: # (B, H, W, C)
             feat = feat.permute(0,3,1,2)
        return self.projection(feat)

class MedFusionNet(nn.Module):
    def __init__(self, num_classes=1, feat_dim=FUSION_DIM, dropout_rate=0.3, pretrained=True, freeze_cnn_blocks=2):
        super().__init__()
        self.local_branch = LocalBranch(pretrained, freeze_cnn_blocks)
        self.global_branch = GlobalBranch(pretrained)
        self.gate_fc = nn.Linear(2*feat_dim, feat_dim)
        self.refine = nn.Sequential(nn.Conv2d(feat_dim, feat_dim, 3, padding=1), nn.BatchNorm2d(feat_dim), nn.ReLU(True))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout_rate), nn.Linear(feat_dim, num_classes))
    def forward(self, x):
        f_c, f_s = self.local_branch(x), self.global_branch(x)
        g = torch.sigmoid(self.gate_fc(torch.cat([f_c.mean([2,3]), f_s.mean([2,3])], 1))).unsqueeze(-1).unsqueeze(-1)
        f_f = self.refine(g*f_c + (1-g)*f_s)
        logit = self.classifier(f_f)
        return {"logit": logit, "prob": torch.sigmoid(logit), "f_c": f_c, "f_s": f_s}
    def count_parameters(self): return {"total": sum(p.numel() for p in self.parameters() if p.requires_grad)}

In [ ]:
%%writefile losses.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class MedFusionLoss(nn.Module):
    def __init__(self, lambda_loc=0.5, lambda_cons=0.3, lambda_cal=0.1, focal_alpha=0.25, focal_gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = focal_alpha, focal_gamma
        self.l_cons, self.l_cal = lambda_cons, lambda_cal
    def forward(self, logits, targets, model_out, cam=None, bboxes=None, bbox_mask=None):
        # Focal Loss
        bce = F.binary_cross_entropy_with_logits(logits.squeeze(), targets.float(), reduction="none")
        p = torch.sigmoid(logits.squeeze())
        pt = p*targets + (1-p)*(1-targets)
        at = self.alpha*targets + (1-self.alpha)*(1-targets)
        l_cls = (at * (1-pt)**self.gamma * bce).mean()
        # Consistency
        l_cons = F.mse_loss(model_out["f_c"].mean([2,3]), model_out["f_s"].mean([2,3]))
        return {"total": l_cls + self.l_cons*l_cons, "cls": l_cls, "cons": l_cons, "loc": torch.tensor(0.0), "cal": torch.tensor(0.0)}

In [ ]:
%%writefile train.py
import os, sys, time, argparse, logging, csv
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from dataset import build_dataloaders; from model import MedFusionNet; from losses import MedFusionLoss

def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loaders = build_dataloaders(args.data_dir, args.batch_size, 384, args.num_workers)
    model = MedFusionNet().to(device)
    criterion = MedFusionLoss(); optimizer = optim.AdamW(model.parameters(), lr=args.lr)
    scaler = GradScaler(enabled=args.amp)
    print(f"Training on {device} with AMP={args.amp}")
    for epoch in range(args.epochs):
        model.train(); total_loss = 0
        for batch in loaders['train']:
            imgs, labs = batch['image'].to(device), batch['label'].to(device)
            optimizer.zero_grad()
            with autocast(enabled=args.amp):
                out = model(imgs); losses = criterion(out['logit'], labs, out)
            scaler.scale(losses['total']).backward(); scaler.step(optimizer); scaler.update()
            total_loss += losses['total'].item()
        print(f"Epoch {epoch+1}/{args.epochs} | Loss: {total_loss/len(loaders['train']):.4f}")
        torch.save(model.state_dict(), f"checkpoints/best.pth")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--data_dir", default="./data"); p.add_argument("--epochs", type=int, default=50)
    p.add_argument("--batch_size", type=int, default=16); p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--num_workers", type=int, default=2); p.add_argument("--amp", action="store_true")
    p.add_argument("--checkpoint_dir", default="./checkpoints"); p.add_argument("--output_dir", default="./outputs")
    train(p.parse_args())

In [ ]:
!pip install -r requirements.txt

In [ ]:
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip -d temp_data
!mkdir -p data
!mv temp_data/chest_xray/train data/
!mv temp_data/chest_xray/val data/
!mv temp_data/chest_xray/test data/
!rm -rf temp_data chest-xray-pneumonia.zip
print("Dataset prêt.")

In [ ]:
!python train.py --data_dir ./data --epochs 50 --batch_size 16 --amp